In [15]:
import pandas as pd
import joblib
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from utils import fit_position_pipeline, select_features_by_correlation

In [16]:
# features = joblib.load('../predictors/hist/feats_def')
features = [
    "was_home",
    "difficulty",
    "opponent_difficulty",
    "win_prob",
    "elo_diff",
    "yellow_cards_rolling_5",
    "starts_rolling_3",
    "points_3_ewm",
    "xP_3_ewm",
    "minutes_rolling_3",
    "goals_scored_3_ewm",
    "assists_3_ewm",
    "clean_sheets_3_ewm",
    "goals_conceded_3_ewm",
    "ict_index_3_ewm",
    "expected_goals_3_ewm",
    "expected_assists_3_ewm",
    "expected_goal_involvements_3_ewm",
    "expected_goals_conceded_3_ewm",
    "interceptions_3_ewm",
    "blocks_3_ewm",
    "clearances_3_ewm",
    "tackles_3_ewm",
    "shots_faced_3_ewm",
    "goals_prevented_3_ewm",
    "selected_by_percent",
    "transfers_in",
]

player_data = pd.read_csv("../../inputs/rolled_data_24-25_xp_fixed.csv")
player_data = player_data[player_data["position"] == "Defender"]

league_models = {}

for gw in range(21, 39):
    print(f"\n{'=' * 60}\n Model for round {gw} \n{'=' * 60}")
    data = player_data[(player_data["round"] > 3) & (player_data["round"] < gw)].fillna(
        0
    )
    feats = features  # features[gw]
    # feats.remove('xP')
    selected_feats = select_features_by_correlation(
        data,
        feats,
        target_col="xP",
        redundancy_threshold=0.4,
        min_corr=0.2,
        verbose=True,
    )

    league_model = fit_position_pipeline(
        data, selected_feats, gw, position_name="DEF", random_seed=42
    )

    print(f"Features for DEF kept in gw {gw} -> {league_model['feats']}")

    league_models[gw] = league_model



 Model for round 21 
  dropping 'xP_3_ewm' (|corr| with xP=0.811) -- |corr|=0.836 with already-selected 'points_3_ewm'
  dropping 'ict_index_3_ewm' (|corr| with xP=0.720) -- |corr|=0.789 with already-selected 'points_3_ewm'
  dropping 'minutes_rolling_3' (|corr| with xP=0.694) -- |corr|=0.761 with already-selected 'points_3_ewm'
  dropping 'starts_rolling_3' (|corr| with xP=0.680) -- |corr|=0.745 with already-selected 'points_3_ewm'
  dropping 'clean_sheets_3_ewm' (|corr| with xP=0.653) -- |corr|=0.824 with already-selected 'points_3_ewm'
  dropping 'expected_goal_involvements_3_ewm' (|corr| with xP=0.578) -- |corr|=0.624 with already-selected 'points_3_ewm'
  dropping 'clearances_3_ewm' (|corr| with xP=0.574) -- |corr|=0.640 with already-selected 'points_3_ewm'
  dropping 'transfers_in' (|corr| with xP=0.552) -- |corr|=0.566 with already-selected 'points_3_ewm'
  dropping 'tackles_3_ewm' (|corr| with xP=0.549) -- |corr|=0.607 with already-selected 'points_3_ewm'
  dropping 'expected_

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 63 seconds.


  divergences: 0
Kept 4 / 4 features: ['elo_diff', 'yellow_cards_rolling_5', 'points_3_ewm', 'goals_conceded_3_ewm']
Refitting on pruned feature set...


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


... (135 outputs left out)

Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 82 seconds.


  divergences: 0
Features for DEF kept in gw 38 -> ['elo_diff', 'points_3_ewm', 'goals_conceded_3_ewm']


In [17]:
joblib.dump(league_models, "./league_models_def")

['./league_models_def']